In [1]:
import random
import pandas as pd
import numpy as np
import io
import base64

In [4]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
T = pd.read_csv("./csv/T.csv",index_col=0)
#T = pd.concat([T.iloc[::10, [0]], T.iloc[::10, 1::10]], axis=1) # slice every 10 rows and column

# for T, just gotta call T.iloc[location1, location2]

patrol_timings = pd.read_csv("./csv/patrol timings.csv",index_col=1)
patrol_timings = patrol_timings.iloc[:,1:]
patrol_timings
# for patrol timings, just gotta call patrol_timings.iloc[location number, 0] for the amount of time
T.index

Index(['Bishan East', 'Marymount', 'Upper Thomson', 'Alexandra Hill',
       'Alexandra North', 'Bukit Ho Swee', 'Bukit Merah', 'Depot Road',
       'Everton Park', 'Henderson Hill',
       ...
       'Serangoon Garden', 'Serangoon North', 'Upper Paya Lebar',
       'Ang Mo Kio Division HQ', 'Bedok Division HQ', 'Central Division HQ',
       'Clementi Division HQ', 'Jurong Division HQ', 'Tanglin Division HQ',
       'Woodlands Division HQ'],
      dtype='object', name='index_column', length=311)

In [ ]:
np.sum(patrol_timings)

In [ ]:
# Find the minimum value in each row of T and sum them up
row_mins = T.min(axis=1)
total_min_sum = row_mins.sum()
print("Sum of row minimums:", total_min_sum)

# this is a lower bound (but too loose)
# i think gurobi's lower bound was 1600+. but this was the first LP relaxation, the lower bound can be better

### Helper Functions (for modularity)

In [ ]:
def patrol_time(location):
    # Add bounds checking
    if location >= len(patrol_timings) or location < 0:
        print(f"Error: patrol_time location {location} out of bounds (max: {len(patrol_timings)-1})")
        return 0
    return patrol_timings.iloc[location, 0]

def travel_time(location1, location2):
    # Add bounds checking
    if location1 >= len(T) or location2 >= len(T.columns) or location1 < 0 or location2 < 0:
        print(f"Error: travel_time locations {location1}, {location2} out of bounds (max: {len(T)-1}, {len(T.columns)-1})")
        return 0
    return T.iloc[location1, location2]

def arrival_time(ls):
    if ls is None or len(ls) < 2 or (len(ls) == 2 and ls[0]==ls[1]):
        return 0
    # sum patrol time location
    total_time = sum(patrol_time(location) for location in ls[1:-1])
    # add travel time for every consevutive pair
    for i in range(len(ls)-1):
        total_time += travel_time(ls[i],ls[i+1])
    return total_time

def latest_arrival_time(encoding):
    return max(arrival_time(ls) for ls in encoding.values() if ls is not None)

def objective_value(encoding):
    objective = 0
    for ls in encoding.values():
        if len(ls) > 2:   
            for i in range(len(ls)-1):  
                    objective += travel_time(ls[i],ls[i+1])
    return objective
        
def feasible(ls): # returns True or False
    return arrival_time(ls) <= 600

def feasible_solution(encoding):
    for ls in encoding.values():
        if not feasible(ls):
            return False
    return True

def no_of_locations(encoding):
    return [len(v)-2 for v in encoding.values() if v is not None]

def summary(encoding):
    print(
        f"Feasible: {feasible_solution(encoding)}\n"
        f"Objective Value: {objective_value(encoding)}\n"
        f"Latest Arrival Time: {latest_arrival_time(encoding)}\n"
        f"No. of Locations Each FRC visits: between {min(no_of_locations(encoding))} and {max(no_of_locations(encoding))}\n"
    )

# Print which vehicles are used, their total time, and their routes\
def route(encoding):
    if feasible_solution(encoding):
        for k in encoding:
            if encoding[k] is None:
                print(f"\nVehicle {k} is not used.")
            else:
                print(f"\nTotal time {k}: {arrival_time(encoding[k])}")
                print(f"Route for vehicle {k}: ", end =" ")
                print(*encoding[k],sep='->')
    else:
        print("❌ No feasible solution found.")

### Map (Folium)

In [ ]:
coordinates = pd.read_csv('./csv/singapore_city_coordinates_improved.csv')
policehqs = pd.read_csv('./csv/policeheadquarters.csv')

coordinates.dropna(subset=['latitude' , 'longitude'], inplace=True)
coordinates = coordinates[(coordinates.Place != 'Park') 
                      & (coordinates.Place != 'Trafalgar') 
                      & (coordinates.Place != 'Semakau')
                      & (coordinates.Place != 'Sudong')
                      & (coordinates.Place != 'Straits View')
                      & (coordinates.Place != 'City Terminals')
                      & (coordinates.Place != 'Murai')
                      & (coordinates.Place != 'Lim Chu Kang')
                      & (coordinates.Place != 'Pulau Seletar')
                      & (coordinates.Place != 'North-Eastern Islands')
                      & (coordinates.Place != 'Pulau Seletar')]
coordinates.reset_index(inplace=True, drop=True)

# Capitalise latitude and longitude headers in coordinates DataFrame
coordinates.rename(columns={'latitude': 'Latitude', 'longitude': 'Longitude'}, inplace=True)


In [ ]:
def map_out(encoding):
    import folium
    import random

    # Assign a color to each HQ
    hq_colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'cadetblue']
    len_locations = coordinates.shape[0]
    hq_indices = list(range(len_locations, len_locations+7))
    hq_color_map = {hq_idx: hq_colors[i % len(hq_colors)] for i, hq_idx in enumerate(hq_indices)}

    # Helper: get location name from index
    def get_location_name(idx):
        if idx < len_locations:
            return T.index[idx]
        else:
            # HQ: get from policehqs DataFrame
            return policehqs.iloc[idx - len_locations, 0]

    # Helper: get lat/lon from name and source
    def get_latlon(idx, name):
        if idx < len_locations:
            row = coordinates.loc[coordinates['Place'] == name]
        else:
            row = policehqs.loc[policehqs['Division'] == name]
        if not row.empty:
            return row.iloc[0]['Latitude'], row.iloc[0]['Longitude']
        return None, None

    # Center map on Singapore
    m = folium.Map(location=[1.3521, 103.8198], zoom_start=12)

    for frc, route in encoding.items():
        if route is not None:
            hq_idx = route[0]
            color = hq_color_map.get(hq_idx, 'black')
            latlons = []
            for idx in route:
                name = get_location_name(idx)
                lat, lon = get_latlon(idx, name)
                if lat is not None and lon is not None:
                    latlons.append((lat, lon))
            # Draw the route
            folium.PolyLine(latlons, color=color, weight=3, opacity=0.7, tooltip=frc).add_to(m)
            # Optionally, mark start/end
            if latlons:
                folium.Marker(latlons[0], icon=folium.Icon(color=color, icon='home'), tooltip=f"{frc} HQ").add_to(m)

    return m



### Solution Encoding
- the solution encoding is a dictionary containing the frc name as the key, and the location number as the values. Numbers range from 0 to 310. 0 to 303 are the patrol locations, 304 to 310 are the hqs.
- eg. encoding = {'hq1_frc1': [305, 223, 283, 77, 252, 197, 305], ...

### Greedy Heuristic Algorithm (Finding a Good Initial Solution)

In [ ]:
def greedy_initial():

    locations_T = pd.read_csv("./csv/df_time.csv" )
    hq_T = pd.read_csv("./csv/hq to locations.csv")
    S = pd.read_csv("./csv/patrol timings.csv", index_col=1 )
    T = pd.concat([locations_T, hq_T], ignore_index=True)
    random.seed(1)
    num_locations = locations_T.shape[0]
    num_hqs = 7
    num_frcs = 56
    frcs_per_hq = num_frcs // num_hqs  # 8 FRCs per HQ
    avg_locations_per_frc = num_locations // num_frcs  # This will be 5 for now
    row_dict = {i: name for i, name in enumerate(T.iloc[:, 0])} # i: name  for each pair in enumerate. enumerate is for all
    hq_names = [row_dict[i] for i in range(num_locations, num_locations+7)]
    frc_names = [f"hq{i+1}_frc{j+1}" for i in range(num_hqs) for j in range(frcs_per_hq)]
    T = np.copy(locations_T.iloc[:,1:].to_numpy())
    HQ = np.copy(hq_T.iloc[:,1:].to_numpy())
    S = S['Amount of time to patrol'].to_numpy()

    encoding = dict.fromkeys(frc_names)
    sum_time = dict.fromkeys(frc_names , 0)
    routes = dict.fromkeys(frc_names , np.array([], dtype=int))
    unavailable_locations = []

    while np.min(T) != 99999:

        for idx, car in enumerate(encoding):

            if routes[car].size == 0: # start of route
                hq_node = int(car[2])-1 # retrieving the HQ number
                p2 = np.where(HQ[hq_node,:] == np.min(HQ[hq_node,:]))[0][0]  # Get nearest location from HQ
                sum_time[car] += HQ[hq_node , p2] + S[p2] # add travelling time from HQ to next and patrolling time on p2

                unavailable_locations.append(p2)

                HQ[:,p2] = 99999 # remove selection from hq to location to prevent selection for other FRC

                routes[car] = np.append(routes[car], hq_node+num_locations) # add HQ node to tour
                routes[car] = np.append(routes[car], p2) # add p2 node to tour

            elif routes[car][0] == routes[car][-1]:
                encoding[car] = list(routes[car])
                pass

            else:
                p1 = routes[car][-1] # retrieve the last / current location
                T_copy = np.copy(T)
                T_copy[:,unavailable_locations] = 99999 
                p2 = np.where(T_copy[p1,:] == np.min(T_copy[p1,:]))[0][0] # find the nearest location from last location
                p1_to_p2 = T_copy[p1 , p2]

                HQ_copy = np.copy(HQ)
                HQ_copy[:,unavailable_locations] = 99999 
                p2_to_hq = HQ_copy[routes[car][0] - num_locations, p2]
                unavailable_locations.append(p2)

                patrol_p2 = S[p2]
                if sum_time[car] + p1_to_p2 + patrol_p2 + p2_to_hq <= 630:  
                    routes[car] = np.append(routes[car], p2) # add p2 node to tour
                    sum_time[car] += p1_to_p2 + patrol_p2
                    T[:,p1] = 99999 
                    T[p1, :] = 99999
                    HQ[:,p1] = 99999 # remove selection from hq to location to prevent selection for other FRC

                else:                    
                    sum_time[car] += HQ[routes[car][0] - num_locations,p1]
                    routes[car] = np.append(routes[car], routes[car][0]) # add p2 node to tour
                    HQ[:,p1] = 99999 # remove selection from hq to location to prevent selection for other FRC
                    T[:,p1] = 99999 
                    T[p1, :] = 99999
                    
    # Convert all numpy arrays to lists and ensure proper format
    final_encoding = {}
    for car in encoding:
        if encoding[car] is None:
            # Convert numpy array to list and ensure all indices are valid
            route_list = [int(x) for x in routes[car] if 0 <= int(x) <= 310]
            final_encoding[car] = route_list
        else:
            final_encoding[car] = encoding[car]
    
    return final_encoding

encoding = greedy_initial()
print(encoding)
summary(encoding)
route(encoding)
map_out(encoding)

In [ ]:
def random_initial_encoding(num_locations=304, num_hqs=7, num_frcs=56):
    import random
    frcs_per_hq = num_frcs // num_hqs  # 8 FRCs per HQ
    avg_locations_per_frc = num_locations // num_frcs  # This will be 5 for now

    # Generate flat list of FRC names
    frc_names = [f"hq{i+1}_frc{j+1}" for i in range(num_hqs) for j in range(frcs_per_hq)]

    # Shuffle locations without modifying the original list
    locations = list(range(num_locations))
    shuffled_locations = random.sample(locations, len(locations))

    # Assign avg_locations_per_frc locations to each FRC
    encoding = {
        frc: shuffled_locations[i*avg_locations_per_frc:(i+1)*avg_locations_per_frc]
        for i, frc in enumerate(frc_names)
    }

    # Distribute the remaining locations randomly (with replacement)
    remaining = shuffled_locations[num_frcs*avg_locations_per_frc:]
    shuffled_frc_names = random.choices(frc_names, k=len(remaining))
    for frc, loc in zip(shuffled_frc_names, remaining):
        encoding[frc].append(loc)

    # Add the HQ index at the front and back of each FRC's route
    for i, (frc, locations) in enumerate(encoding.items()):
        hq_index = num_locations + (i // frcs_per_hq)  # 8 FRCs per HQ, HQ indices: 304, 305, ..., 311
        encoding[frc] = [hq_index] + locations + [hq_index]

    return encoding


### the solution encoding dictionary
- the solution encoding is a dictionary where the key is the name of each FRC eg. hq1_frc1, and the value is a list of location indices assigned to that FRC
- example: encoding = {'hq1_frc1': [305, 223, 283, 77, 252, 197, 305], ...

### Local Search Operators
- These operators are used for Local Search (Hill Climbing) Algorithm

In [ ]:
def swap_operator(encoding):
    """
    Randomly selects two FRCs and swaps one location between them (not HQs).
    Only performs the swap if both resulting routes are feasible and the swap improves the objective.
    This version simulates the swap on copies first, only updating the original if accepted.
    """

    frc_list = [frc for frc in encoding if len(encoding[frc]) > 2]
    if len(frc_list) < 2:
        return

    frc1, frc2 = random.sample(frc_list, 2)
    idx1 = random.randint(1, len(encoding[frc1]) - 2)
    idx2 = random.randint(1, len(encoding[frc2]) - 2)

    # Simulate swap on copies
    new1 = encoding[frc1][:]
    new2 = encoding[frc2][:]
    new1[idx1], new2[idx2] = new2[idx2], new1[idx1]

    # Check feasibility and improvement
    if feasible(new1) and feasible(new2):
        old_obj = objective_value({frc1: encoding[frc1], frc2: encoding[frc2]})
        new_obj = objective_value({frc1: new1, frc2: new2})
        if new_obj < old_obj:
            encoding[frc1] = new1
            encoding[frc2] = new2
            return True

def insertion_operator(encoding):
    """
    Randomly selects a location from one FRC and inserts it before a random location in another FRC (can be the same FRC).
    Only performs the insertion if both resulting routes are feasible AND the move improves the objective function.
    This version simulates the move on copies first, only updating the original if accepted.
    """

    frc_list = [frc for frc in encoding if len(encoding[frc]) > 3]
    if not frc_list:
        return

    frc_from, frc_to = random.sample(frc_list, 2) if len(frc_list) > 1 else (frc_list[0], frc_list[0])

    idx_from = random.randint(1, len(encoding[frc_from]) - 2)
    loc = encoding[frc_from][idx_from]

    # Pick a random insertion point in the target FRC (not at start)
    insert_pos = random.randint(1, len(encoding[frc_to]) - 1)

    # If same FRC and insert_pos is right after idx_from, skip (no change)
    if frc_from == frc_to and (insert_pos == idx_from or insert_pos == idx_from + 1):
        return

    # Simulate move on copies
    new_from = encoding[frc_from][:]  # copy
    new_to = encoding[frc_to][:]      # copy

    # Remove from source
    del new_from[idx_from]
    # Insert into target
    new_to.insert(insert_pos, loc)

    # Check feasibility and improvement
    if feasible(new_from) and feasible(new_to):
        old_obj = objective_value({frc_from: encoding[frc_from], frc_to: encoding[frc_to]})
        new_obj = objective_value({frc_from: new_from, frc_to: new_to})
        if new_obj < old_obj:
            encoding[frc_from] = new_from
            encoding[frc_to] = new_to
            return True

In [ ]:
def two_opt_operator(encoding):
    frc_list = [frc for frc in encoding if encoding[frc] is not None and len(encoding[frc]) >= 4] # checks that there are at least 4 elements in the list, so that 2-opt is possible
    if not frc_list:
        return
    frc = random.choice(frc_list)
    route = encoding[frc]
    i, j = sorted(random.sample(range(1, len(route) - 1), 2)) # samples between the 2nd element and 2nd last element
    new_route = route[:i] + route[i:j+1][::-1] + route[j+1:] # reverses the route between i and j
    if feasible(new_route):
        old_obj = objective_value({frc: route})
        new_obj = objective_value({frc: new_route})
        if new_obj <= old_obj:
            encoding[frc] = new_route
            return True
        
# What: Reverse a segment within a single FRC's route.
# How: Pick one FRC, pick two indices i < j, reverse the segment between them.
# Why: Removes route "crossings" and can greatly improve a single FRC's path.

In [ ]:
def cross_exchange_operator(encoding):
    """
    Cross-exchange (interroute 2-opt):
    Selects two FRCs and exchanges a segment from each route.
    Considers all 4 segment orderings (original/reversed) and applies the best feasible improvement.
    """

    frc_list = [frc for frc in encoding if encoding[frc] is not None and len(encoding[frc]) >= 4]
    if len(frc_list) < 2: # essentially checks if there are enough frcs to even swap
        return

    frc1, frc2 = random.sample(frc_list, 2) # pick 2 frcs from the list
    route1, route2 = encoding[frc1], encoding[frc2]

    # Randomly choose the segment 
    start1, end1 = sorted(random.sample(range(1, len(route1)-1),2)) # samples between 2nd element and 2nd last element
    start2, end2 = sorted(random.sample(range(1, len(route2)-1),2))
    
    # Extract segments
    seg1 = route1[start1:end1+1] 
    seg2 = route2[start2:end2+1]

    # All 4 combinations: (seg1, seg2), (seg1[::-1], seg2), (seg1, seg2[::-1]), (seg1[::-1], seg2[::-1])
    candidates = []
    for s1 in [seg1, seg1[::-1]]: 
        for s2 in [seg2, seg2[::-1]]: # permutates
            new1 = route1[:start1] + s2 + route1[end1+1:]
            new2 = route2[:start2] + s1 + route2[end2+1:]
            if feasible(new1) and feasible(new2):
                old_obj = objective_value({frc1: route1, frc2: route2})
                new_obj = objective_value({frc1: new1, frc2: new2})
                if new_obj < old_obj:
                    candidates.append((new_obj, new1, new2))

    if candidates:
        # Pick the best improvement
        candidates.sort(key=lambda x: x[0])
        _, best_new1, best_new2 = candidates[0]
        encoding[frc1] = best_new1
        encoding[frc2] = best_new2
        return True


In [ ]:
def or_opt_operator(encoding, max_chain_length=3):
    """
    Interroute Or-opt operator with Simulated Annealing:
    Moves a chain of 1 to max_chain_length consecutive locations
    from one FRC's route to a different position in another FRC's route.
    Tries both original and reversed chain, accepts worse solutions with SA probability.
    """
    
    max_chain_length = 3
    frc_list = [frc for frc in encoding if encoding[frc] is not None and len(encoding[frc]) > max_chain_length + 1]
    if len(frc_list) < 2:
        return

    frc_from, frc_to = random.sample(frc_list, 2)
    route_from = encoding[frc_from]
    route_to = encoding[frc_to]
    n_from = len(route_from)
    n_to = len(route_to)

    # Only select chains that do not include HQs (i.e., indices 1 to n_from-2)
    if n_from <= 3:
        return  # not enough locations to move

    chain_len = random.randint(1, min(max_chain_length, n_from - 3))
    start = random.randint(1, n_from - 1 - chain_len)
    end = start + chain_len

    # Make sure chain does not include HQs
    if start == 0 or end > n_from - 1:
        return

    # Possible insertion points in target route (not at HQ, i.e., 1 to n_to-1)
    possible_inserts = list(range(1, n_to))  # can't insert at 0 (HQ) or n_to (HQ)
    if not possible_inserts:
        return

    insert_pos = random.choice(possible_inserts)
    chain = route_from[start:end]

    # Remove chain from source
    new_from_base = route_from[:start] + route_from[end:]

    # Try both original and reversed chain
    candidates = []
    for candidate_chain in [chain, chain[::-1]]:
        new_to = route_to[:insert_pos] + candidate_chain + route_to[insert_pos:]
        # Ensure HQs are still at start and end
        if new_from_base[0] >= 305 and new_from_base[-1] >= 305 and new_to[0] >= 305 and new_to[-1] >= 305:
            if feasible(new_from_base) and feasible(new_to):
                old_obj = objective_value({frc_from: route_from, frc_to: route_to})
                new_obj = objective_value({frc_from: new_from_base, frc_to: new_to})
                if new_obj < old_obj:
                    candidates.append((new_obj, new_from_base, new_to))

    if candidates:
        # Pick the best (lowest objective) among accepted moves
        candidates.sort(key=lambda x: x[0])
        _, best_from, best_to = candidates[0]
        encoding[frc_from] = best_from
        encoding[frc_to] = best_to
        return True

# SA

In [ ]:
def temp_func(T, iter):
    return T/float(1+iter), "T/float(1+iter)"

def temp_probaility(T0,  delta_E):
    import math
    return math.exp(-(delta_E/T0))

In [ ]:

def swap_operator_SA(encoding ,T):
    """
    Randomly selects two FRCs and swaps one location between them (not HQs).
    Only performs the swap if both resulting routes are feasible and the swap improves the objective.
    This version simulates the swap on copies first, only updating the original if accepted.
    """

    frc_list = [frc for frc in encoding if encoding[frc] is not None and len(encoding[frc]) > 2] 
    if len(frc_list) < 2: # this wont actl happen, but its basically to check if there are 2 frcs in the frc_list, otherwise a swap wouldnt be possible
        return

    frc1, frc2 = random.sample(frc_list, 2)
    idx1 = random.randint(1, len(encoding[frc1]) - 2)
    idx2 = random.randint(1, len(encoding[frc2]) - 2)

    # Simulate swap on copies
    new1 = encoding[frc1][:]
    new2 = encoding[frc2][:]
    new1[idx1], new2[idx2] = new2[idx2], new1[idx1]

    # Check feasibility and improvement
    if feasible(new1) and feasible(new2):
        old_obj = objective_value({frc1: encoding[frc1], frc2: encoding[frc2]}) # uses the objective_value(encoding) function
        new_obj = objective_value({frc1: new1, frc2: new2})
        if new_obj < old_obj:
            encoding[frc1] = new1
            encoding[frc2] = new2
            return True
        else:
            # If new_obj is worse than old_obj, accept new_obj with a probability        
            acceptance_rate = random.uniform(0,1)
            if temp_probaility(T, new_obj - old_obj) > acceptance_rate:
                encoding[frc1] = new1
                encoding[frc2] = new2
                return True

In [ ]:
def insertion_operator_SA(encoding,T):
    """
    Randomly selects a location from one FRC and inserts it before a random location in another FRC (can be the same FRC).
    Only performs the insertion if both resulting routes are feasible AND the move improves the objective function.
    This version simulates the move on copies first, only updating the original if accepted.
    """

    frc_list = [frc for frc in encoding if len(encoding[frc]) > 3] # check for frcs that visit at least 2 locations
    if not frc_list: 
        return

    frc_from, frc_to = random.sample(frc_list, 2) if len(frc_list) > 1 else (frc_list[0], frc_list[0])

    idx_from = random.randint(1, len(encoding[frc_from]) - 2)
    loc = encoding[frc_from][idx_from]

    # Pick a random insertion point in the target FRC (not at start)
    insert_pos = random.randint(1, len(encoding[frc_to]) - 1)

    # If same FRC and insert_pos is right after idx_from, skip (no change)
    if frc_from == frc_to and (insert_pos == idx_from or insert_pos == idx_from + 1):
        return

    # Simulate move on copies
    new_from = encoding[frc_from][:]  # copy
    new_to = encoding[frc_to][:]      # copy

    # Remove from source
    del new_from[idx_from]
    # Insert into target
    new_to.insert(insert_pos, loc)

    # Check feasibility and improvement
    if feasible(new_from) and feasible(new_to):
        old_obj = objective_value({frc_from: encoding[frc_from], frc_to: encoding[frc_to]})
        new_obj = objective_value({frc_from: new_from, frc_to: new_to})
        if new_obj < old_obj:
            encoding[frc_from] = new_from
            encoding[frc_to] = new_to
            return True
        else:
            # If new_obj is worse than old_obj, accept new_obj with a probability        
            acceptance_rate = random.uniform(0,1)
            if temp_probaility(T, new_obj - old_obj) > acceptance_rate:
                encoding[frc_from] = new_from
                encoding[frc_to] = new_to
                return True

In [ ]:
def two_opt_operator_SA(encoding,T): # segment reverse
    frc_list = [frc for frc in encoding if encoding[frc] is not None and len(encoding[frc]) >= 4] # checks that there are at least 4 elements in the list, so that 2-opt is possible
    if not frc_list:
        return
    frc = random.choice(frc_list)
    route = encoding[frc]
    i, j = sorted(random.sample(range(1, len(route) - 1), 2)) # samples between the 2nd element and 2nd last element
    new_route = route[:i] + route[i:j+1][::-1] + route[j+1:] # reverses the route between i and j
    if feasible(new_route):
        old_obj = objective_value({frc: route})
        new_obj = objective_value({frc: new_route})
        if new_obj <= old_obj:
            encoding[frc] = new_route
            return True
        else:
            # If new_obj is worse than old_obj, accept new_obj with a probability        
            acceptance_rate = random.uniform(0,1)
            if temp_probaility(T, new_obj - old_obj) > acceptance_rate:
                encoding[frc] = new_route
                #print("2opt worse:", new_obj-old_obj, new_obj, old_obj, i, j, route, new_route)
                return True
        
        
# What: Reverse a segment within a single FRC's route.
# How: Pick one FRC, pick two indices i < j, reverse the segment between them.
# Why: Removes route "crossings" and can greatly improve a single FRC's path.

In [ ]:

def cross_exchange_operator_SA(encoding,T): # segment swap between 2 frcs
    """
    Cross-exchange (interroute 2-opt):
    Selects two FRCs and exchanges a segment from each route.
    Considers all 4 segment orderings (original/reversed) and applies the best feasible improvement.
    """

    frc_list = [frc for frc in encoding if encoding[frc] is not None and len(encoding[frc]) >= 4]
    if len(frc_list) < 2: # essentially checks if there are enough frcs to even swap
        return

    frc1, frc2 = random.sample(frc_list, 2) # pick 2 frcs from the list
    route1, route2 = encoding[frc1], encoding[frc2]

    # Randomly choose the segment 
    start1, end1 = sorted(random.sample(range(1, len(route1)-1),2)) # samples between 2nd element and 2nd last element
    start2, end2 = sorted(random.sample(range(1, len(route2)-1),2))
    
    # Extract segments
    seg1 = route1[start1:end1+1] 
    seg2 = route2[start2:end2+1]

    # All 4 combinations: (seg1, seg2), (seg1[::-1], seg2), (seg1, seg2[::-1]), (seg1[::-1], seg2[::-1])
    candidates = []
    for s1 in [seg1, seg1[::-1]]: 
        for s2 in [seg2, seg2[::-1]]: # permutates
            new1 = route1[:start1] + s2 + route1[end1+1:]
            new2 = route2[:start2] + s1 + route2[end2+1:]
            if feasible(new1) and feasible(new2):
                old_obj = objective_value({frc1: route1, frc2: route2})
                new_obj = objective_value({frc1: new1, frc2: new2})
                if new_obj < old_obj:
                    candidates.append((new_obj, new1, new2))
                else:
                    acceptance_rate = random.uniform(0,1)
                    if temp_probaility(T, new_obj - old_obj) > acceptance_rate:
                        candidates.append((new_obj, new1, new2))

    if candidates:
        # Pick the best improvement
        candidates.sort(key=lambda x: x[0])
        _, best_new1, best_new2 = candidates[0]
        encoding[frc1] = best_new1
        encoding[frc2] = best_new2
        return True


In [ ]:
def or_opt_operator_SA(encoding, T):  # segment insert
    """
    Interroute Or-opt operator with Simulated Annealing:
    Moves a chain of 1 to max_chain_length consecutive locations
    from one FRC's route to a different position in another FRC's route.
    Tries both original and reversed chain, accepts worse solutions with SA probability.
    """

    max_chain_length = 3
    frc_list = [frc for frc in encoding if encoding[frc] is not None and len(encoding[frc]) > max_chain_length + 1]
    if len(frc_list) < 2:
        return

    frc_from, frc_to = random.sample(frc_list, 2)
    route_from = encoding[frc_from]
    route_to = encoding[frc_to]
    n_from = len(route_from)
    n_to = len(route_to)

    # Only select chains that do not include HQs (i.e., indices 1 to n_from-2)
    if n_from <= 3:
        return  # not enough locations to move

    chain_len = random.randint(1, min(max_chain_length, n_from - 3))
    start = random.randint(1, n_from - 1 - chain_len)
    end = start + chain_len

    # Make sure chain does not include HQs
    if start == 0 or end > n_from - 1:
        return

    # Possible insertion points in target route (not at HQ, i.e., 1 to n_to-1)
    possible_inserts = list(range(1, n_to))  # can't insert at 0 (HQ) or n_to (HQ)
    if not possible_inserts:
        return

    insert_pos = random.choice(possible_inserts)
    chain = route_from[start:end]

    # Remove chain from source
    new_from_base = route_from[:start] + route_from[end:]

    # Try both original and reversed chain
    candidates = []
    for candidate_chain in [chain, chain[::-1]]:
        new_to = route_to[:insert_pos] + candidate_chain + route_to[insert_pos:]
        # Ensure HQs are still at start and end
        if new_from_base[0] >= 305 and new_from_base[-1] >= 305 and new_to[0] >= 305 and new_to[-1] >= 305:
            if feasible(new_from_base) and feasible(new_to):
                old_obj = objective_value({frc_from: route_from, frc_to: route_to})
                new_obj = objective_value({frc_from: new_from_base, frc_to: new_to})
                if new_obj < old_obj:
                    candidates.append((new_obj, new_from_base, new_to))
                else:
                    # Simulated Annealing acceptance
                    acceptance_rate = random.uniform(0, 1)
                    if temp_probaility(T, new_obj - old_obj) > acceptance_rate:
                        candidates.append((new_obj, new_from_base, new_to))

    if candidates:
        # Pick the best (lowest objective) among accepted moves
        candidates.sort(key=lambda x: x[0])
        _, best_from, best_to = candidates[0]
        encoding[frc_from] = best_from
        encoding[frc_to] = best_to
        return True

In [ ]:
def run_SA(encoding, time_limit=600, elapsed_time=0, record_progress=True):
    import time
    import random
    
    start_time = time.time()
    last_print_time = start_time
    swap_count = 0
    insertion_count = 0
    two_opt_count = 0
    cross_count = 0
    or_opt_count = 0
    T_iter = 0
    iter = 0 # iterations
    T_iter_max = 30 # update temp every x iteration
 
    T_max = 1500 # intial temp
    T = T_max 
    times = []
    objectives = []
    iters = []
    temp = []


    while (time.time() - start_time + elapsed_time) < time_limit: # 10 minutes
        
        if T_iter > T_iter_max:
            T , Tfunc_str = temp_func(T_max, iter)
            T_iter = 0

        probability_selection = random.random()
        if probability_selection < 0.2:
            if two_opt_operator_SA(encoding, T):
                two_opt_count += 1

        elif probability_selection < 0.4:
            if insertion_operator_SA(encoding, T):
                insertion_count += 1

        elif probability_selection < 0.6:
            if swap_operator_SA(encoding, T):
                swap_count += 1

        elif probability_selection < 0.8:
            if cross_exchange_operator_SA(encoding, T):
                cross_count += 1

        elif probability_selection < 1:
            if or_opt_operator_SA(encoding, T):
                or_opt_count +=1
    
        T_iter += 1
        iter += 1
        
        # Record every 10 seconds
        if time.time() - last_print_time >= 10:
            total_elapsed = int(time.time() - start_time + elapsed_time) # time.time() - start time is the time this function has been running. elapsed time is from the previous
            obj = objective_value(encoding)
            print(
                f"{total_elapsed}s, Objective value: {obj}, "
                f"Temperature: {T} "
                f"Swaps: {swap_count}, "
                f"Insertions: {insertion_count}, "
                f"Two opt: {two_opt_count}, "
                f"Cross: {cross_count}, "
                f"Or opt: {or_opt_count}, "
                f"Iterations :{iter}"
            )            
            last_print_time = time.time()
            if record_progress:
                times.append(total_elapsed)
                objectives.append(obj)
                iters.append(iter)
                temp.append(T)
    total_time = elapsed_time + (time.time() - start_time)
    if record_progress:
        # Also record the final value
        times.append(int(total_time))
        objectives.append(objective_value(encoding))
        iters.append(iter)
        temp.append(T)
        return encoding, objective_value(encoding), total_time, times, objectives, iters, temp, (swap_count, insertion_count,two_opt_count,cross_count,or_opt_count),T_max, T_iter_max, Tfunc_str
    else:
        return encoding, objective_value(encoding), total_time



In [ ]:
def run_localsearch(encoding, time_limit=600, elapsed_time=0, record_progress=True):
    import time
    import random

    start_time = time.time()
    last_print_time = start_time
    swap_count = 0
    insertion_count = 0
    two_opt_count = 0
    cross_count = 0
    or_opt_count = 0
 
    times = []
    objectives = []

    while (time.time() - start_time + elapsed_time) < time_limit: # 10 minutes
        probability_selection = random.random()

        if probability_selection < 0.2:
            if swap_operator(encoding):
                swap_count += 1
        elif probability_selection < 0.4:
            if insertion_operator(encoding):
                insertion_count += 1
        elif probability_selection < 0.6:
            if two_opt_operator(encoding):
                two_opt_count += 1
        elif probability_selection < 0.8:
            if cross_exchange_operator(encoding):
                cross_count += 1
        elif probability_selection < 1.0:
            if or_opt_operator(encoding):
                or_opt_count +=1 

        # Record every 10 seconds
        if time.time() - last_print_time >= 10:
            total_elapsed = int(time.time() - start_time + elapsed_time) # time.time() - start time is the time this function has been running. elapsed time is from the previous
            obj = objective_value(encoding)
            print(
                f"{total_elapsed}s, Objective value: {obj}, "
                f"Swaps: {swap_count}, "
                f"Insertions: {insertion_count}, "
                f"Two opt: {two_opt_count}, "
                f"Cross: {cross_count}, "
                f"Or opt: {or_opt_count}"
            )            
            last_print_time = time.time()
            if record_progress:
                times.append(total_elapsed)
                objectives.append(obj)
    total_time = elapsed_time + (time.time() - start_time)
    if record_progress:
        # Also record the final value
        times.append(int(total_time))
        objectives.append(objective_value(encoding))
        return encoding, objective_value(encoding), total_time, times, objectives
    else:
        return encoding, objective_value(encoding), total_time




In [ ]:
def plot_objective(times, objectives): # plots the objective function over time
    # Plot (outside the function)
    import matplotlib.pyplot as plt
    import io

    plt.plot(times, objectives, marker='o')
    plt.xlabel('Elapsed Time (s)')
    plt.ylabel('Objective Value')
    plt.title('Objective Value Over Time')
    plt.grid(True)
    buf = io.BytesIO()
    plt.savefig(buf, format='png')

    buf.seek(0)
    plt.show()
    plt.close()
    img_base64 = base64.b64encode(buf.read()).decode('utf-8')
    buf.close()

    return img_base64

### Main Function

In [ ]:
from IPython.display import display
def main_greedy_initial_SA(time_limit=600):
    
    # Generate initial encoding
    encoding = greedy_initial()
    
    # Print summary of initial encoding
    summary(encoding)
    
    # Run metaheuristics
    encoding, obj, total_time, times, objectives, iters, temp, search_opt_counts, T_initial, T_iter_max , Tfunc_str= run_SA(encoding, time_limit=time_limit, elapsed_time=0, record_progress=True)
    
    # Print final summary
    summary(encoding)
    
    # Plot objective value over time
    obj_plot = plot_objective(times, objectives)
    
    # Plot temp over iter
    convergence_plot = plot_convergence(iters, temp)

    # Saving Matplotlib graph as an image
    
    # Map out the routes
    m = map_out(encoding)
    display(m)

    # Save map to a temporary HTML string
    map_html = m._repr_html_()  # or use folium_map.get_root().render()

    # Write to HTML file
    filename = get_timestamped_filename("tsp_SAgreedy_report")
    with open(filename, "w") as f:
        f.write(writehtml(search_opt_counts , T_initial, T_iter_max, Tfunc_str , time_limit, encoding, obj_plot, convergence_plot, map_html))

    return encoding, obj, total_time, times, objectives

def main_random_initial_localsearch(time_limit=600):
    # Generate initial encoding
    encoding = random_initial_encoding() # random_initial_encoding()
    
    # Print summary of initial encoding
    summary(encoding)
    
    # Run metaheuristics
    encoding, obj, total_time, times, objectives = run_localsearch(encoding, time_limit=time_limit, elapsed_time=0, record_progress=True)
    
    # Print final summary
    summary(encoding)
    
    # Plot objective value over time
    plot_objective(times, objectives)
    
    # Map out the routes
    m = map_out(encoding)
    display(m)
    # m.save("frc_routes.html")

    return encoding, obj, total_time, times, objectives

def continue_solving(encoding_tuple, time_limit=1200):
    # its for me to continue solving the encoding, after first runnning it in main, if it hasn't reached objective
    # time_limit is the total time (including previous), to run for

    encoding = encoding_tuple[0]
    elapsed_time = encoding_tuple[2]
    
    summary(encoding)

    # Continue solving with the existing encoding
    encoding, obj, total_time, times, objectives = run_localsearch(encoding, time_limit=time_limit, elapsed_time=elapsed_time, record_progress=True)
    
    # Print final summary
    print(encoding)
    summary(encoding)
    
    # Plot objective value over time
    plot_objective(times, objectives)
    
    # Map out the routes
    m = map_out(encoding)
    display(m)
    # m.save("frc_routes.html")

    return encoding, obj, total_time

# encoding_tuple3 = main_random_initial(time_limit=1800)
# # encoding_tuple stores the final encoding, its objective value, and time taken.

In [ ]:
def plot_convergence(iter, temp): # plots the objective function over time
    # Plot (outside the function)
    import matplotlib.pyplot as plt
    import io

    plt.plot(iter, temp, marker='o')
    plt.xlabel('Iterations')
    plt.ylabel('Temperature')
    plt.title('Convergence')
    plt.grid(True)
    buf = io.BytesIO()
    plt.savefig(buf, format='png')

    buf.seek(0)
    plt.show()
    plt.close()
    img_base64 = base64.b64encode(buf.read()).decode('utf-8')
    buf.close()

    return img_base64

In [ ]:
def writehtml(p1, p2, p3, p4, p5, p6, p7, p8,p9):

    return f"""
    <html>
    <body>
        <h1>TSP Report</h1>
        <h3>Parameters</h3>
            <p>Selection of Neighbourhood: Probabilistic Selecction</p>
            <p>Number of Swap operators: {p1[0]}</p>
                <ul>
                <li>Acceptance rate: Random</li>
                </ul>
            <p>Number of Insertion operators: {p1[1]}</p>
                <ul>
                <li>Acceptance rate: Random</li>
                </ul> 
            <p>Number of 2-opt operators: {p1[2]}</p> 
                <ul>
                <li>Acceptance rate: Random</li>
                </ul>
            <p>Number of Cross operators: {p1[3]}</p> 
                <ul>
                <li>Acceptance rate: Random</li>
                </ul>
            <p>Number of Or-Opt operators: {p1[4]}</p> 
                <ul>
                <li>Acceptance rate: Random</li>
                </ul>
            <p>Initial Temperature:{p2} </p>
            <p>Update Temperature After {p3} iterations</p>
            <p>Update Temperature function using "{p4}"</p>

        <h3>Results</h3>
            <p>Time Limit: {p5}</p>
            <p>Feasible: {feasible_solution(p6)}</p>
            <p>Objective Value: {objective_value(p6)}</p>
            <p>Latest Arrival Time: {latest_arrival_time(p6)}</p>
            <p>No. of Locations Each FRC visits: between {min(no_of_locations(p6))} and {max(no_of_locations(p6))}</p>
        <h3>Final Solution</h3>
            {''.join(f"<p>Total time for {key}: {arrival_time(value)}&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Route:{value}</p>" for key, value in p6.items())}
        <h3>Matplotlib Plot</h3>
            <img src="data:image/png;base64,{p7}"/>
        <h3>Convergence Plot</h3>
            <img src="data:image/png;base64,{p8}"/>
        <h3>Folium Map</h3>
        {p9}
    </body>
    </html>
    """
    

In [ ]:
def get_timestamped_filename(prefix="report", extension="html"):
    from datetime import datetime

    now = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{prefix}_{now}.{extension}"


### Run SA with Greedy Initial Solution

In [ ]:
encoding_tuple4 = main_greedy_initial_SA(1800)

In [ ]:
encoding_tuple4
summary(encoding_tuple4[0])
objective_value(encoding_tuple4[0])

### Find duplicate values

In [ ]:
values = list(encoding_tuple4[0].values())
flat_list = [
    x
    for xs in values
    for x in xs
]

import collections
print([item for item, count in collections.Counter(flat_list).items() if count > 1])

### Run Local Search with Random Initial Solution

In [ ]:
encoding_tuple5 = main_random_initial_localsearch(1800)

### Location Assignment Algorithm (using K means) for another good initial solution

In [ ]:
distance_matrix_1 = T.iloc[:-7, -7:]
distance_matrix_1

In [ ]:
import numpy as np
nearest_hq = np.argmin(distance_matrix_1, axis=1)
print(nearest_hq)

np.bincount(nearest_hq)



In [ ]:
assignment = {}
for i in range(7):
    assignment[i] = [] # where i is the hq index

for i in range(len(nearest_hq)):
    assignment[nearest_hq[i]].append(i)
    
# assignment is a dictionary with 0-6 as the keys, and a list of locations as the values

In [ ]:
from sklearn.cluster import KMeans
import numpy as np

encoding_kmeans = {}

for hq, loc_list in assignment.items():
    if not loc_list:
        continue
    n_clusters = 8
    # Get coordinates for these locations
    coords = coordinates.set_index('Place').loc[T.index[loc_list]][['Latitude', 'Longitude']].values

    # Run KMeans with one random init for more exploration
    kmeans = KMeans(n_clusters=n_clusters, n_init=1, random_state=None)
    labels = kmeans.fit_predict(coords)

    # Group locations by cluster label
    clusters = [[] for _ in range(n_clusters)]
    for loc, label in zip(loc_list, labels):
        clusters[label].append(loc)

    # Balance clusters to be as equal as possible (+-1)
    # Flatten and reassign if needed
    all_locs = [loc for cluster in clusters for loc in cluster]
    avg = len(loc_list) // n_clusters
    rem = len(loc_list) % n_clusters
    balanced_clusters = []
    start = 0
    for i in range(n_clusters):
        end = start + avg + (1 if i < rem else 0)
        balanced_clusters.append(all_locs[start:end])
        start = end

    # Store in encoding format
    for i, cluster in enumerate(balanced_clusters):
        frc_name = f"hq{hq+1}_frc{i+1}"
        hq_index = 304 + hq
        encoding_kmeans[frc_name] = [hq_index] + cluster + [hq_index]

In [ ]:
map_out(encoding_kmeans)

In [ ]:
summary(encoding_kmeans)

In [ ]:
#run_metaheuristics(encoding_kmeans)